In [2]:
import os

import cv2
import openslide
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm

In [3]:
DATA_DIR = "../data/"
CSV_PATH = os.path.join(DATA_DIR, "train.csv")
IMAGE_DIR = os.path.join(DATA_DIR, "train_images")
MASK_DIR = os.path.join(DATA_DIR, "train_label_masks")

In [4]:
df = pd.read_csv(CSV_PATH)
df.head()

,image_id,data_provider,isup_grade,gleason_score
0,0005f7aaab2800f6170c399693a96917,karolinska,0,0+0
1,000920ad0b612851f8e01bcc880d9b3d,karolinska,0,0+0
2,0018ae58b01bdadc8e347995b69f99aa,radboud,4,4+4
3,001c62abd11fa4b57bf7a6c603a11bb9,karolinska,4,4+4
4,001d865e65ef5d2579c190a0e0350d8f,karolinska,0,0+0


In [5]:
print(df.shape)

(10616, 4)


In [6]:
keys = [
    "slide_id",
    "gleason_score",
    "isup_grade",
    "provider",
    "slide_exists",
    "mask_exists",
    "feature_file_exists",
    "feature_tile_count",
    "tiff_pym_lvls",
    "tiff_lvl0_dim",
    "tiff_lvl_ds",
    "tiff_mpx",
    "percentage_empty",
]

In [7]:
row, col = df.shape

manifest_df = pd.DataFrame(index=range(df.shape[0]), columns=keys)

for i in tqdm.tqdm(range(row)):
    new_row = {a: None for a in keys}
    new_row[keys[0]] = df.loc[i]["image_id"]
    gleason_score = df.loc[i]["gleason_score"]
    if gleason_score == "negative":
        new_row[keys[1]] = "0+0"
    else:
        new_row[keys[1]] = gleason_score

    new_row[keys[2]] = df.loc[i]["isup_grade"]
    new_row[keys[3]] = df.loc[i]["data_provider"]
    new_row[keys[4]] = os.path.exists(os.path.join(IMAGE_DIR, f"{new_row[keys[0]]}.tiff"))
    new_row[keys[5]] = os.path.exists(os.path.join(MASK_DIR, f"{new_row[keys[0]]}_mask.tiff"))
    new_row[keys[6]] = None
    new_row[keys[7]] = None

    try:
        slide = openslide.OpenSlide(os.path.join(IMAGE_DIR, f"{new_row[keys[0]]}.tiff"))
        new_row[keys[8]] = slide.level_count
        new_row[keys[9]] = slide.level_dimensions[0]
        new_row[keys[10]] = slide.level_downsamples
        new_row[keys[11]] = (
            float(slide.properties.get("openslide.mpp-x")),
            float(slide.properties.get("openslide.mpp-y")),
        )
        image = slide.read_region((0, 0), slide.level_count - 1, slide.level_dimensions[slide.level_count - 1])
        image = cv2.cvtColor(np.array(image), cv2.COLOR_RGBA2GRAY)
        histogram, _ = np.histogram(image, bins=8, range=(0, 256))
        histogram = histogram / histogram.sum()
        histogram = histogram.tolist()
        percentage_empty = histogram[-1] * 100
        new_row[keys[12]] = percentage_empty

        slide.close()
    except Exception as e:
        print(f"Slide: {new_row[keys[0]]}  ;\n Exception: {e}")

    manifest_df.loc[i] = new_row

100%|██████████| 10616/10616 [07:27<00:00, 23.71it/s]


In [8]:
manifest_df.head()

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
0,0005f7aaab2800f6170c399693a96917,0+0,0,karolinska,True,True,None,None,3,"(27648, 29440)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",96.895192
1,000920ad0b612851f8e01bcc880d9b3d,0+0,0,karolinska,True,True,None,None,3,"(15360, 13312)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",94.424705
2,0018ae58b01bdadc8e347995b69f99aa,4+4,4,radboud,True,True,None,None,3,"(5888, 25344)","(1.0, 4.0, 16.0)","(0.4861876369654638, 0.4861876369654638)",82.860363
3,001c62abd11fa4b57bf7a6c603a11bb9,4+4,4,karolinska,True,True,None,None,3,"(23904, 28664)","(1.0, 4.0, 16.00223338916806)","(0.5031982437947761, 0.5031982437947761)",95.209238
4,001d865e65ef5d2579c190a0e0350d8f,0+0,0,karolinska,True,True,None,None,3,"(28672, 34560)","(1.0, 4.0, 16.0)","(0.45201826153776614, 0.45201826153776614)",94.389209


In [9]:
manifest_df.shape[0] == row

True

In [10]:
manifest_df["mask_exists"].value_counts()

mask_exists
True     10516
False      100
Name: count, dtype: int64

In [11]:
invalid_slides_opened = manifest_df[manifest_df["tiff_pym_lvls"] == None].shape[0]
if invalid_slides_opened == 0:
    print("All slides opened correctly")
else:
    print(f"{invalid_slides_opened} slides incorrectly")

All slides opened correctly


In [12]:
manifest_df["tiff_pym_lvls"].value_counts()

tiff_pym_lvls
3    10616
Name: count, dtype: int64

In [13]:
downsampling_distribution = manifest_df["tiff_lvl_ds"].apply(lambda x: tuple([int(i) for i in x]))
downsampling_distribution.value_counts()

tiff_lvl_ds
(1, 4, 16)    10616
Name: count, dtype: int64

In [14]:
manifest_df["tiff_lvl0_dim"].min()

(1280, 2304)

In [15]:
resolutions_radboud = manifest_df[manifest_df["provider"] == "radboud"]["tiff_mpx"].to_list()
resolutions_karolinska = manifest_df[manifest_df["provider"] == "karolinska"]["tiff_mpx"].tolist()
res_r_x, res_r_y = zip(*resolutions_radboud)
res_k_x, res_k_y = zip(*resolutions_karolinska)
df_r_x = pd.Series(res_r_x)
df_r_x.describe()

count    5.160000e+03
mean     4.861876e-01
std      1.110331e-16
min      4.861876e-01
25%      4.861876e-01
50%      4.861876e-01
75%      4.861876e-01
max      4.861876e-01
dtype: float64

In [16]:
df_k_x = pd.Series(res_k_x)
df_k_x.describe()

count    5456.000000
mean        0.472590
std         0.025095
min         0.452018
25%         0.452018
50%         0.452018
75%         0.503198
max         0.503198
dtype: float64

In [17]:
df_r_y = pd.Series(res_r_y)
df_r_y.describe()

count    5.160000e+03
mean     4.861876e-01
std      1.110331e-16
min      4.861876e-01
25%      4.861876e-01
50%      4.861876e-01
75%      4.861876e-01
max      4.861876e-01
dtype: float64

In [18]:
df_k_y = pd.Series(res_k_y)
df_k_y.describe()

count    5456.000000
mean        0.472590
std         0.025095
min         0.452018
25%         0.452018
50%         0.452018
75%         0.503198
max         0.503198
dtype: float64

In [19]:
gleason_grades = {
    "0+0": np.int64(0),
    "3+3": np.int64(1),
    "3+4": np.int64(2),
    "4+3": np.int64(3),
    "4+4": np.int64(4),
    "5+3": np.int64(4),
    "3+5": np.int64(4),
    "4+5": np.int64(5),
    "5+4": np.int64(5),
    "5+5": np.int64(5),
}

In [20]:
manifest_df[manifest_df["isup_grade"] != manifest_df["gleason_score"].apply(gleason_grades.get)]

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
7273,b0a92a74cb53899311acc30b7405e101,4+3,2,karolinska,True,True,None,None,3,"(20916, 35512)","(1.0, 4.0, 16.00333283567217)","(0.5031982437947761, 0.5031982437947761)",94.980748


In [24]:
manifest_df.to_parquet(os.path.join(DATA_DIR, "derived/slide_inventory.parquet"), index=False)